### Setup & Config

In [0]:
from pyspark.sql.functions import trim, col, when

# ERP tables config
ERP_TABLES = {
    "cust_az12": {
        "bronze": "`databricks-medallion-lakehouse`.bronze.cust_az12",
        "silver": "`databricks-medallion-lakehouse`.silver.cust_az12",
        "renames": {
            "CID": "customer_id_erp",
            "BDATE": "birthdate",
            "GEN": "gender"
        }
    },
    "loc_a101": {
        "bronze": "`databricks-medallion-lakehouse`.bronze.loc_a101",
        "silver": "`databricks-medallion-lakehouse`.silver.loc_a101",
        "renames": {
            "CID": "customer_id_erp",
            "CNTRY": "country"
        }
    },
    "px_cat_g1v2": {
        "bronze": "`databricks-medallion-lakehouse`.bronze.px_cat_g1v2",
        "silver": "`databricks-medallion-lakehouse`.silver.px_cat_g1v2",
        "renames": {
            "ID": "category_id",
            "CAT": "category",
            "SUBCAT": "subcategory",
            "MAINTENANCE": "maintenance_required"
        }
    }
}

print("="*70)
print("SILVER: ERP Tables (cust_az12 + loc_a101 + px_cat_g1v2)")
print("="*70)

### Process All 3 ERP Tables (Loop)

In [0]:
# Process all 3 ERP tables
for table_name, config in ERP_TABLES.items():
    bronze_table = config["bronze"]
    silver_table = config["silver"]
    renames = config["renames"]
    
    print(f"\n{'='*70}")
    print(f"Processing: {table_name}")
    print(f"{'='*70}")
    
    # Step 1: Read from Bronze
    df = spark.read.table(bronze_table)
    rows_before = df.count()
    print(f"  Bronze rows: {rows_before:,}")
    
    # Step 2: Trim all string columns
    df_clean = df.select([
        trim(col(c)).alias(c) if df.schema[c].dataType.simpleString() == "string" else col(c)
        for c in df.columns
    ])
    
    # Step 3: Rename columns using config
    for old_name, new_name in renames.items():
        df_clean = df_clean.withColumnRenamed(old_name, new_name)
    
    # Step 4: Deduplicate
    # Get first column as primary key (varies by table)
    primary_key = list(renames.values())[0]  # First renamed column is the key
    df_clean = df_clean.dropDuplicates([primary_key])
    rows_after = df_clean.count()
    
    # Step 5: Write to Silver
    spark.sql(f"DROP TABLE IF EXISTS {silver_table}")
    
    df_clean.write \
        .mode("overwrite") \
        .format("delta") \
        .saveAsTable(silver_table)
    
    print(f"  Silver rows: {rows_after:,}")
    print(f"  ✅ Written to: {silver_table}")

print(f"\n{'='*70}")
print("✅ ALL ERP TABLES COMPLETE")
print(f"{'='*70}")